/**
 * CbDinquiry.gs – Automated Assessment Inquiry System (Bound Script)
 * -------------------------------------------------------------------------
 * ‑ This script is designed to be BOUND to the Google Sheet containing
 * the 'CbDinquiry', 'CbD', 'Residents', and 'ProgramDirector' sheets.
 * ‑ Triggered by form submission to the 'CbDinquiry' sheet.
 * ‑ Looks up student email and name from the 'Residents' sheet (using CbD structure).
 * ‑ Searches the 'CbD' sheet for all assessments matching the student's Roll Number.
 * ‑ Creates an HTML table summarizing the found assessments from the 'CbD' sheet.
 * ‑ Sends an email with the table to the inquirer (from form) and CCs the student.
 * ‑ Includes fallback logic in onInquiryFormSubmit for manual testing or missing event object.
 *
 * HOW TO INSTALL
 * 1.  Open the Google Sheet that contains your 'CbDinquiry', 'CbD', 'Residents',
 * and 'ProgramDirector' sheets.
 * 2.  Go to Extensions > Apps Script. This will create a NEW script project BOUND
 * to this specific spreadsheet.
 * 3.  Paste this entire code into the script editor (replace any default code).
 * 4.  Save the script.
 * 5.  Set up an installable On-form-submit trigger:
 * - Go to the Triggers menu (clock icon on the left).
 * - Click '+ Add Trigger'.
 * - Choose `onInquiryFormSubmit` as the function to run.
 * - Choose deployment to run: Select `Head`.
 * - Select event source: Choose `From spreadsheet`.
 * - Select event type: Choose `On form submit`.
 * - Click 'Save'. You may be asked to authorize the script (this authorization is for
 * the bound script to access the spreadsheet it's in and send emails).
 * 6.  Ensure your sheets ('CbDinquiry', 'CbD', 'Residents') have the correct names
 * and column structures as used in the CbD automation script.
 * -------------------------------------------------------------------------*/

In [ ]:
// --- Column Maps (0-based index for script arrays) ---
// CbDinquiry Sheet Columns
const INQUIRY_COL_TIMESTAMP = 0;
const INQUIRY_COL_EMAIL = 1; // Inquirer Email
const INQUIRY_COL_STUDENT_ROLL = 2; // Student Roll Number

// Residents Sheet Columns (matching CbD script structure)
const RESIDENTS_COL_NAME = 0;      // Column A: Full Name
const RESIDENTS_COL_EMAIL = 1;     // Column B: Email Address
const RESIDENTS_COL_ROLL_NUMBER = 2; // Column C: Roll Number

// CbD Sheet Columns (matching CbD script structure)
const CBD_COL_TIMESTAMP = 0; // Column A
const CBD_COL_FACULTY_EMAIL = 1; // Column B
const CBD_COL_RESIDENT_ROLL = 2; // Column C - Used for matching
const CBD_COL_YEAR = 3;      // Column D
const CBD_COL_DATE = 4;      // Column E
const CBD_COL_SPECIALITY = 5; // Column F
const CBD_COL_TOPIC = 6;     // Column G
const CBD_COL_EPA_NUMBER = 7; // Column H
const CBD_COL_CASE_DESCRIPTION = 8; // Column I (Used as Procedure Name in inquiry)
const CBD_COL_ASSESSOR_POSITION = 19; // Column T (Assessor's Position)
// Analytics columns start at U (20 in 0-based index) in CbD script
const CBD_COL_FULL_MARK = 20; // Column U
const CBD_COL_TOTAL_MARK = 21; // Column V
const CBD_COL_PERCENTAGE = 22; // Column W
const CBD_COL_MEAN = 23; // Column X
const CBD_COL_SD = 24; // Column Y
const CBD_COL_VARIANCE = 25; // Column Z
const CBD_COL_DISCRIMINATION = 26; // Column AA
const CBD_COL_NARRATIVE_SKILL = 27; // Column AB
// Overall Clinical Care is Column P (15) in CbD script, used as Overall ability in inquiry
const CBD_COL_OVERALL_CARE = 15; // Column P
const CBD_COL_AREAS_DONE_WELL = 16; // Column Q (Areas done well)
const CBD_COL_ACTION_NEEDED = 17; // Column R (Specific action needed)


/* === ON‑SUBMIT HANDLER FOR INQUIRY FORM (ROBUST FOR BOUND SCRIPT) ======= */
/**
 * Triggered by a form submission to the 'CbDinquiry' sheet.
 * Processes the inquiry and sends the assessment summary email.
 * Includes fallback for manual running/missing event object.
 * @param {Object} e The event object from the form submission (may be undefined if run manually).
 */
function onInquiryFormSubmit(e) {
  try {
    Logger.log('onInquiryFormSubmit triggered');

    let inquirySheet, row, values;
    const ss = SpreadsheetApp.getActiveSpreadsheet(); // Get the bound spreadsheet

    // Check if event object and range are available (typical for form submit trigger)
    if (e && e.range) {
      Logger.log('Event object found with range. Processing directly.');
      inquirySheet = e.range.getSheet();
      row = e.range.getRow();
      values = e.range.getValues()[0]; // Get values from the submitted row

       // Basic check to ensure it's the Inquiry sheet, although trigger should handle this
       if (inquirySheet.getName() !== 'CbDinquiry') {
           Logger.log("Trigger fired on a sheet other than CbDinquiry. Skipping.");
           return;
       }


    } else {
      Logger.log('No event object or range found. Assuming manual run or trigger issue. Using fallback.');
      // Fallback for manual running or if the trigger didn't provide 'e.range'
      inquirySheet = ss.getSheetByName("CbDinquiry");

      if (!inquirySheet) {
        Logger.log('CbDinquiry sheet not found using fallback.');
        return;
      }

      row = inquirySheet.getLastRow();
       // Get values from the last row, assuming it's the one to process
       const lastCol = inquirySheet.getLastColumn();
       if (row <= 1 || lastCol < INQUIRY_COL_STUDENT_ROLL + 1) { // Check if there's data beyond header and enough columns
            Logger.log('CbDinquiry sheet is empty or has insufficient data in the last row for fallback processing.');
            return;
       }
      values = inquirySheet.getRange(row, 1, 1, lastCol).getValues()[0];
       Logger.log(`Fallback: Retrieved values from last row (${row}) on sheet ${inquirySheet.getName()}`);
    }


    Logger.log(`Processing inquiry row ${row} on sheet ${inquirySheet.getName()}`);

    // Check if row is valid (must be greater than 1 to avoid header)
    if (row <= 1) {
      Logger.log('Skipping header row or invalid row number: ' + row);
      return;
    }

    // Extract necessary info from the submitted row using column constants
    const inquirerEmail = values[INQUIRY_COL_EMAIL];
    const studentRollNumber = values[INQUIRY_COL_STUDENT_ROLL];

    if (!inquirerEmail || !studentRollNumber) {
        Logger.log("Inquirer email or student roll number is missing in the row data. Cannot process inquiry.");
        return;
    }

    Logger.log(`Inquiry from: ${inquirerEmail}, for student roll: ${studentRollNumber}`);

    // Get student name and email (using the bound spreadsheet)
    const studentInfo = getStudentInfoByRollNumber(studentRollNumber, ss); // Pass the active spreadsheet object
    Logger.log(`Student Info Lookup: ${JSON.stringify(studentInfo)}`);

    // Create the assessment summary table (using the bound spreadsheet)
    const assessmentTableHtml = createAssessmentTable(studentRollNumber, ss); // Pass the active spreadsheet object
    Logger.log("Assessment table created.");

    // Send the email
    sendInquiryEmail(inquirerEmail, studentInfo, studentRollNumber, assessmentTableHtml);

    Logger.log('onInquiryFormSubmit completed successfully.');

  } catch (error) {
    Logger.log('Error in onInquiryFormSubmit: ' + error.toString());
    if (error.stack) {
      Logger.log('Stack trace: ' + error.stack);
    }
    // Optionally send an error notification to an admin
    // MailApp.sendEmail('admin@example.com', 'Assessment Inquiry Script Error', 'Error processing inquiry:\n' + error.toString() + '\nStack: ' + error.stack);
  }
}


/* === LOOKUP HELPERS (Adapted for Bound Script) =================================== */

/**
 * Looks up student name and email in the 'Residents' sheet by Roll Number,
 * using the column structure defined in the CbD script (A=Name, B=Email, C=Roll).
 * Works within the bound spreadsheet.
 * @param {string|number} rollNumber The student's roll number to look up.
 * @param {SpreadsheetApp.Spreadsheet} ss The active bound spreadsheet object.
 * @returns {{studentName: string, studentEmail: string}} An object containing the student's name and email, or empty strings if not found.
 */
function getStudentInfoByRollNumber(rollNumber, ss) {
  try {
    if (!rollNumber) {
      Logger.log('getStudentInfoByRollNumber: No roll number provided');
      return { studentName: "", studentEmail: "" };
    }

    // Use getSheetByName on the active bound spreadsheet
    const sh = ss.getSheetByName('Residents'); // Use CbD Residents sheet name

    if (!sh) {
      Logger.log('Residents sheet not found in the bound spreadsheet');
      return { studentName: "", studentEmail: "" };
    }

    const dataRange = sh.getDataRange();
    if (!dataRange || dataRange.getValues().length <= 1) {
        Logger.log('Residents sheet is empty or only has headers');
        return { studentName: "", studentEmail: "" };
    }
    const data = dataRange.getValues();

    // Normalize the roll number for comparison (case-insensitive, no leading/trailing spaces)
    const normalizedRollNumber = String(rollNumber).trim().toLowerCase();

    // Loop through data starting from the second row (index 1)
    for (let i = 1; i < data.length; i++) {
      const row = data[i];
      // Ensure the row has enough columns and the roll number column (Col C, index 2) is not empty
      if (!row || row.length <= RESIDENTS_COL_ROLL_NUMBER || !row[RESIDENTS_COL_ROLL_NUMBER]) continue;

      const rowRollNumber = String(row[RESIDENTS_COL_ROLL_NUMBER]).trim().toLowerCase(); // Compare against Column C (index 2)

      if (rowRollNumber === normalizedRollNumber) {
        Logger.log(`Student found in Residents: Name: "${row[RESIDENTS_COL_NAME] || ''}", Email: "${row[RESIDENTS_COL_EMAIL] || ''}"`);
        return {
          studentName: row[RESIDENTS_COL_NAME] || '',    // Get Name from Column A (index 0)
          studentEmail: row[RESIDENTS_COL_EMAIL] || ''   // Get Email from Column B (index 1)
        };
      }
    }

    Logger.log(`Student not found in Residents for roll number: "${rollNumber}"`);
    return { studentName: "", studentEmail: "" }; // not found
  } catch (error) {
    Logger.log('Error in getStudentInfoByRollNumber: ' + error.toString());
    if (error.stack) {
      Logger.log('Stack trace: ' + error.stack);
    }
    return { studentName: "", studentEmail: "" };
  }
}


/* === TABLE CREATION (Adapted for Bound Script) =================================== */

/**
 * Creates an HTML table summarizing assessments for a given roll number from the 'CbD' sheet.
 * Extracts data based on the CbD column map. Works within the bound spreadsheet.
 * @param {string|number} rollNumber The student's roll number to filter assessments by.
 * @param {SpreadsheetApp.Spreadsheet} ss The active bound spreadsheet object.
 * @returns {string} An HTML string representing the table.
 */
function createAssessmentTable(rollNumber, ss) {
  try {
    if (!rollNumber) {
      Logger.log('createAssessmentTable: No roll number provided');
      return '<p>No roll number provided for assessment lookup.</p>';
    }

    // Use getSheetByName on the active bound spreadsheet
    const cbdSheet = ss.getSheetByName('CbD'); // Use CbD sheet name

    if (!cbdSheet) {
      Logger.log('CbD sheet not found in the bound spreadsheet');
      return '<p>CbD sheet not found.</p>';
    }

    const dataRange = cbdSheet.getDataRange();
     if (!dataRange || dataRange.getValues().length <= 1) {
        Logger.log('CbD sheet is empty or only has headers');
        return '<p>No assessment data available in the CbD sheet.</p>';
    }
    const cbdData = dataRange.getValues(); // Get all data including headers

    const tableRows = [];

    // Normalize the roll number for comparison
    const normalizedRollNumber = String(rollNumber).trim().toLowerCase();

    // Loop through data starting from the second row (index 1) to find matching assessments
    for (let i = 1; i < cbdData.length; i++) {
      const row = cbdData[i];

      // Ensure the row has enough columns to contain all the data we intend to read from the CbD sheet
      // based on the maximum column index used in the tableRow extraction below.
       const maxColumnIndexNeeded = Math.max(
          CBD_COL_TIMESTAMP,
          CBD_COL_FACULTY_EMAIL,
          CBD_COL_RESIDENT_ROLL,
          CBD_COL_YEAR,
          CBD_COL_DATE,
          CBD_COL_SPECIALITY,
          CBD_COL_TOPIC,
          CBD_COL_EPA_NUMBER,
          CBD_COL_CASE_DESCRIPTION,
          CBD_COL_ASSESSOR_POSITION,
          CBD_COL_TOTAL_MARK,
          CBD_COL_PERCENTAGE,
          CBD_COL_OVERALL_CARE,
          CBD_COL_AREAS_DONE_WELL,
          CBD_COL_ACTION_NEEDED
      ); // No +1 needed here, just the max index

      if (!row || row.length <= maxColumnIndexNeeded || !row[CBD_COL_RESIDENT_ROLL]) {
           Logger.log(`Skipping row ${i+1} due to insufficient columns or missing roll number.`);
           continue;
      }


      const rowResidentRoll = String(row[CBD_COL_RESIDENT_ROLL]).trim().toLowerCase(); // Compare against Column C (index 2)

      if (rowResidentRoll === normalizedRollNumber) {
        // Extract data for the table row using CbD column constants
        const tableRow = [
          row[CBD_COL_TIMESTAMP] || '', // Timestamp (Column A)
          row[CBD_COL_FACULTY_EMAIL] || '', // Evaluating Faculty Email (Column B)
          row[CBD_COL_ASSESSOR_POSITION] || '', // Assessor Position (Column T)
          row[CBD_COL_YEAR] || '', // Year (Column D)
          row[CBD_COL_DATE] || '', // Date of assessment (Column E)
          row[CBD_COL_SPECIALITY] || '', // Speciality (Column F)
          row[CBD_COL_TOPIC] || '', // EPA Topic (Column G)
          row[CBD_COL_EPA_NUMBER] || '', // EPA Number (Column H)
          row[CBD_COL_CASE_DESCRIPTION] || '', // Case Description (Column I)
          row[CBD_COL_TOTAL_MARK] || '', // TotalMark (Column V)
          row[CBD_COL_PERCENTAGE] || '', // Percentage (Column W)
          row[CBD_COL_OVERALL_CARE] || '', // Overall Clinical Care (Column P)
          row[CBD_COL_AREAS_DONE_WELL] || '', // Areas done well (Column Q)
          row[CBD_COL_ACTION_NEEDED] || '' // Action needed (Column R)
        ];
         // Format Date if it's a Date object
        if (tableRow[4] instanceof Date) {
            tableRow[4] = tableRow[4].toLocaleDateString();
        }
         // Format Percentage if it's a number
        if (typeof tableRow[10] === 'number') {
            tableRow[10] = tableRow[10].toFixed(1) + '%';
        }


        tableRows.push(tableRow);
      }
    }

    if (tableRows.length === 0) {
        return `<p>No assessments found for roll number "${rollNumber}".</p>`;
    }

    // Build the HTML table
    let html = '<table border="1" style="border-collapse: collapse; width: 100%;">';
    // Table Headers - Adjusted to match data extracted
    html += '<tr><th>TimeStamp</th><th>Evaluating Faculty Email</th><th>Assessor Position</th><th>Year</th>' +
              '<th>Date of assessment</th><th>Speciality</th><th>Topic</th><th>EPA Number</th>' +
              '<th>Case Description</th><th>Total Mark</th><th>Percentage</th>' +
              '<th>Overall Clinical Care</th><th>Areas done well</th><th>Action needed</th></tr>';

    // Table Rows
    tableRows.forEach(row => {
      html += '<tr>' + row.map(cell => `<td>${cell}</td>`).join('') + '</tr>';
    });
    html += '</table>';

    return html;
  } catch (error) {
    Logger.log('Error in createAssessmentTable: ' + error.toString());
    if (error.stack) {
      Logger.log('Stack trace: ' + error.stack);
    }
    return '<p>Error retrieving assessment data.</p>';
  }
}


/* === EMAIL DISPATCH =================================================== */

/**
 * Sends the assessment inquiry email to the inquirer and CCs the student.
 * @param {string} inquirerEmail The email address of the person who made the inquiry.
 * @param {{studentName: string, studentEmail: string}} studentInfo Object containing student name and email.
 * @param {string|number} studentRollNumber The student's roll number.
 * @param {string} assessmentTableHtml The HTML string of the assessment summary table.
 */
function sendInquiryEmail(inquirerEmail, studentInfo, studentRollNumber, assessmentTableHtml) {
  try {
    const studentName = studentInfo.studentName || "Resident";
    const studentEmail = studentInfo.studentEmail || "";

    // Recipients: To the inquirer, CC the student if email is found
    const recipients = [inquirerEmail];
    if (studentEmail) {
        recipients.push(studentEmail);
    }

    if (recipients.length === 0) {
        Logger.log('No valid recipient email addresses found for inquiry email. Skipping.');
        return;
    }

    const subject = `Assessment Inquiry for ${studentName}`; // Based on your description
    const body = `Inquiry made by ${inquirerEmail}<br>` +
                 `Inquiry for assessment of ${studentName} roll number ${studentRollNumber}<br><br>` +
                 `${assessmentTableHtml}<br><br>` + // Insert the HTML table
                 `This email is sent to the inquirer and CC'd to the student.<br><br>` + // Clarified who receives the email
                 `Thank you<br>Dean Office`; // Closing as requested

    MailApp.sendEmail({
      to: inquirerEmail, // Explicitly send to inquirer
      cc: studentEmail, // Explicitly CC student (will be ignored if studentEmail is empty)
      subject: subject,
      htmlBody: body // Use htmlBody to render the HTML table
    });

    Logger.log(`Inquiry email sent successfully to: ${inquirerEmail}${studentEmail ? ', CC: ' + studentEmail : ''}`);

  } catch(err) {
    Logger.log('Error sending inquiry email: ' + err.toString());
    if (err.stack) {
      Logger.log('Stack trace: ' + err.stack);
    }
  }
}

/* === MANUAL TESTING FUNCTION (Adapted for Bound Script) =================================== */
/**
 * A helper function to manually test the inquiry process using the last row
 * of the 'CbDinquiry' sheet within the bound spreadsheet.
 */
function testCbDinquiry() {
  Logger.log("Starting manual test of Assessment Inquiry system (Bound Script)...");
  try {
    const ss = SpreadsheetApp.getActiveSpreadsheet(); // Get the bound spreadsheet
    const inquirySheet = ss.getSheetByName("CbDinquiry");

    if (!inquirySheet || inquirySheet.getLastRow() <= 1) {
      Logger.log("CbDinquiry sheet not found or is empty (beyond header). Cannot test.");
      return;
    }

    const lastRow = inquirySheet.getLastRow();
    const lastCol = inquirySheet.getLastColumn();
    const values = inquirySheet.getRange(lastRow, 1, 1, lastCol).getValues()[0];

    // Simulate the event object structure needed by onInquiryFormSubmit
    // Note: For manual testing, the 'range' property isn't strictly necessary now
    // due to the fallback logic in onInquiryFormSubmit, but including it
    // makes the simulation closer to a real trigger event.
    const simulatedEvent = {
      range: inquirySheet.getRange(lastRow, 1, 1, lastCol), // Simulate range
      source: ss, // Simulate source spreadsheet (the bound one)
      values: values, // Simulate values
      authMode: ScriptApp.AuthMode.LIMITED // Simulate auth mode
    };

    // Call the main handler function with the simulated event
    onInquiryFormSubmit(simulatedEvent);

    Logger.log("Manual test of Assessment Inquiry system completed. Check logs and emails.");

  } catch (error) {
    Logger.log("Error during manual test: " + error.toString());
    if (error.stack) {
      Logger.log("Stack trace: " + error.stack);
    }
  }
}

